# 03 Recommendations and evaluation

Owner: Jeevan Mandala (ML Engineer)

This is truncated SVD with the k from notebook 02, scored in two ways.

Cosine is what the brief asks for in ML3. A product's vector is its row of V scaled to length 1, and a customer's score for a product is the sum of its cosine similarity to the products they bought, weighted by purchase value.

Reconstruct is the final model. It takes the customer's row of the rank-k matrix, which keeps how popular each product is. We added it once cosine lost to the baseline on the validation period (notebook 02).

Products a customer already bought are never recommended. The baseline is the 10 best-selling training products the customer hasn't bought, where best seller means the most different buyers. Hit rate at 10 is the share of evaluation customers who have at least one held-out product in their top 10. The test purchases are only used here, once, after k and the scoring were settled on the validation period.

In [9]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path

import numpy as np
import pandas as pd

from recommend import (
    load_processed, fit_svd, product_vectors, similar_products_all,
    recommend_for_customers, recommend_blend, best_seller_baseline,
    hits_at_n, bootstrap_ci, paired_bootstrap_diff, nightly_job, TOP_N,
)

A, customer_index, product_index, eval_pairs = load_processed()
CHOSEN_K = int(Path("../outputs/results/chosen_k.txt").read_text())  # from notebook 02
print("Matrix:", A.shape, "| k =", CHOSEN_K)

Matrix: (5471, 3686) | k = 100


## Fit the model

In [10]:
svd = fit_svd(A, CHOSEN_K)
P = product_vectors(svd)
print("Product vectors:", P.shape)

Product vectors: (3686, 100)


## Similar products

The neighbours of the three best sellers, as a sanity check.

In [11]:
neighbours = similar_products_all(P)
popularity = np.asarray((A > 0).sum(axis=0)).ravel()
description = product_index["Description"].to_numpy()

for j in np.argsort(-popularity)[:3]:
    print(f"\nProduct: {description[j]}  (bought by {popularity[j]} customers)")
    for neighbour in neighbours[j]:
        print("   -", description[neighbour])


Product: WHITE HANGING HEART T-LIGHT HOLDER  (bought by 1422 customers)
   - RED HANGING HEART T-LIGHT HOLDER
   - CANDLEHOLDER PINK HANGING HEART
   - RED HANGING METAL STAR TLIGHT HLDR
   - COLOUR GLASS. STAR T-LIGHT HOLDER
   - JAZZ HEARTS SPIRAL NOTEPAD
   - CREAM HEART CARD HOLDER
   - SCALLOP SHELL SOAP DISH
   - GLASS STAR FROSTED T-LIGHT HOLDER
   - EDWARDIAN 2 DRAWER WALL CABINET
   - SILVER LARIAT 40CM

Product: REGENCY CAKESTAND 3 TIER  (bought by 1213 customers)
   - ROSES REGENCY TEACUP AND SAUCER
   - GREEN REGENCY TEACUP AND SAUCER
   - PINK REGENCY TEACUP AND SAUCER
   - REGENCY TEAPOT ROSES
   - SET OF 3 REGENCY CAKE TINS
   - REGENCY MILK JUG PINK
   - REGENCY TEA PLATE ROSES
   - REGENCY SUGAR BOWL GREEN
   - REGENCY TEA PLATE PINK
   - REGENCY TEA PLATE GREEN

Product: PACK OF 72 RETRO SPOT CAKE CASES  (bought by 1058 customers)
   - SET OF 72 RETRO SPOT PAPER  DOILIES
   - PACK OF 72 SKULL CAKE CASES
   - PACK OF 60 PINK PAISLEY CAKE CASES
   - 60 TEATIME FAIRY CA

## Hit rate at 10

In [12]:
eval_rows = np.sort(eval_pairs["customer_idx"].unique())
per_customer = eval_pairs.groupby("customer_idx").size()
print("Evaluation customers:", len(eval_rows))
print("Held-out products per customer: mean", round(per_customer.mean(), 1),
      "| median", int(per_customer.median()))

COSINE = f"SVD cosine, ML3 as specified (k={CHOSEN_K})"
FINAL = f"SVD reconstruct, final model (k={CHOSEN_K})"
BASELINE = "Best-seller baseline"

recs = {
    COSINE: recommend_for_customers(A, P, eval_rows),
    FINAL: recommend_blend(A, svd, eval_rows, mode="reconstruct", alpha=1.0),
    BASELINE: best_seller_baseline(A, eval_rows),
}
hits = {name: hits_at_n(r, eval_rows, eval_pairs, A.shape) for name, r in recs.items()}

rows = []
for name, h in hits.items():
    low, high = bootstrap_ci(h)
    rows.append({"method": name, "hit_rate_at_10": h.mean(), "ci_low": low, "ci_high": high})
results = pd.DataFrame(rows).round(4)
results

Evaluation customers: 1913
Held-out products per customer: mean 23.5 | median 15


,method,hit_rate_at_10,ci_low,ci_high
0,"SVD cosine, ML3 as specified (k=100)",0.2253,0.2059,0.2426
1,"SVD reconstruct, final model (k=100)",0.3832,0.3617,0.4072
2,Best-seller baseline,0.2703,0.2504,0.2896


Both are scored on the same customers, so the fair test is the paired difference: resample the customers, take the difference between the two hit rates each time, and see whether the 95% interval excludes 0.

In [13]:
diff_rows = []
for name in (COSINE, FINAL):
    diff, low, high = paired_bootstrap_diff(hits[name], hits[BASELINE])
    diff_rows.append({"model": name, "hit_rate_minus_baseline": diff, "ci_low": low, "ci_high": high})
paired = pd.DataFrame(diff_rows).round(4)
paired

,model,hit_rate_minus_baseline,ci_low,ci_high
0,"SVD cosine, ML3 as specified (k=100)",-0.0450,-0.0721,-0.0219
1,"SVD reconstruct, final model (k=100)",0.1129,0.0863,0.1396


In [14]:
# customers hit by only one of the two (final model vs baseline)
model_hits, baseline_hits = hits[FINAL], hits[BASELINE]
print("Both hit:        ", int((model_hits & baseline_hits).sum()))
print("Only SVD hits:   ", int((model_hits & ~baseline_hits).sum()))
print("Only baseline:   ", int((~model_hits & baseline_hits).sum()))
print("Neither:         ", int((~model_hits & ~baseline_hits).sum()))

Both hit:         281
Only SVD hits:    452
Only baseline:    236
Neither:          944


## Nightly job timing

Fit, similar products for every product, and top 10 for every customer, for both scorings.

In [15]:
timing = pd.DataFrame([
    nightly_job(A, CHOSEN_K, mode="cosine"),
    nightly_job(A, CHOSEN_K, mode="reconstruct"),
])
timing

,k,mode,fit_seconds,similar_products_seconds,all_customers_seconds,total_seconds
0,100,cosine,0.28,0.2,0.36,0.83
1,100,reconstruct,0.27,0.2,0.42,0.89


In [16]:
results.to_csv("../outputs/results/hit_rate_results.csv", index=False)
paired.to_csv("../outputs/results/paired_differences.csv", index=False)
timing.to_csv("../outputs/results/nightly_job_timing.csv", index=False)

## What this shows

The final model beats the best sellers, 0.383 against 0.270. The paired difference is +0.113 with a 95% interval of 0.086 to 0.140, which doesn't include 0. 452 customers are hit only by the model and 236 only by the baseline.

The cosine scoring loses (0.225, so 0.045 below the baseline, interval -0.072 to -0.022). Scaling the product vectors to length 1 throws away how popular a product is. This business reorders popular lines, and the test period is the run up to Christmas, so what customers newly buy tends to be popular. Reconstruct keeps that weight and still uses the customer's own history.

The neighbours look sensible. The Regency cakestand sits next to the Regency teacups, plates and teapot, hanging heart holders sit next to other heart and star holders, and retro spot cake cases sit next to other cake cases and doilies. A few entries at the end of some lists are weaker. That supports the "customers who bought this also bought" box.

The whole nightly job takes about 0.88 s at k = 100.

Some limits. Customers with no history (13,840 held-out pairs) and products with fewer than 10 buyers (4,725 pairs) can't be served, and the counts are in `split_report.csv`. It's also one test period only (9 October to 9 December 2011), and purchases with no customer ID are invisible.